# DML Lab Assignment No. 5
## Part A: RNN for Text Classification (IMDB Sentiment)
## Part B: LSTM for Time Series Prediction

---
## PART A — RNN for Sentiment Classification (IMDB Dataset)

### Step 1: Import Libraries and Load Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

print('TensorFlow version:', tf.__version__)

NUM_WORDS  = 10000   # Keep top 10k most frequent words
MAX_LEN    = 200     # Pad/truncate sequences to 200 tokens

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=NUM_WORDS)
print(f'Train: {len(X_train)} reviews | Test: {len(X_test)} reviews')

### Step 2: Normalize / Pad Sequences

In [ ]:
X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)

### Step 3: Build the RNN Model

In [ ]:
rnn_model = models.Sequential([
    layers.Embedding(NUM_WORDS, 64, input_length=MAX_LEN),
    layers.SimpleRNN(64, return_sequences=False),
    layers.Dense(32, activation='relu'),
    layers.Dense(1,  activation='sigmoid')   # Binary: positive / negative
])

rnn_model.summary()

### Step 4: Compile the RNN Model

In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Step 5: Train the RNN Model

In [ ]:
rnn_history = rnn_model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

### Step 6: Evaluate / Predict

In [ ]:
rnn_loss, rnn_acc = rnn_model.evaluate(X_test, y_test, verbose=0)
print(f'RNN Test Accuracy: {rnn_acc*100:.2f}%')
print(f'RNN Test Loss:     {rnn_loss:.4f}')

### Step 7: Visualize RNN Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
    ax.plot(rnn_history.history[metric], label=f'Train {title}')
    ax.plot(rnn_history.history[f'val_{metric}'], label=f'Val {title}')
    ax.set_title(f'RNN {title}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

---
## PART B — LSTM for Time Series Prediction (Sine Wave)

### Step 1: Generate Time Series Dataset

In [ ]:
# Generate a noisy sine wave as synthetic time series
np.random.seed(42)
t      = np.linspace(0, 100, 1000)
series = np.sin(t) + 0.1 * np.random.randn(len(t))

plt.figure(figsize=(12, 3))
plt.plot(t, series)
plt.title('Synthetic Time Series (Noisy Sine Wave)')
plt.xlabel('Time')
plt.ylabel('Value')
plt.grid(True)
plt.show()

### Step 2: Prepare Sequences & Normalize

In [ ]:
from sklearn.preprocessing import MinMaxScaler

SEQ_LEN = 30   # Use 30 past steps to predict next step

scaler = MinMaxScaler()
scaled = scaler.fit_transform(series.reshape(-1, 1))

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X_ts, y_ts = make_sequences(scaled, SEQ_LEN)

split = int(len(X_ts) * 0.8)
X_tr, X_te = X_ts[:split], X_ts[split:]
y_tr, y_te = y_ts[:split], y_ts[split:]

print(f'Train sequences: {X_tr.shape}, Test sequences: {X_te.shape}')

### Step 3 & 4: Build the LSTM Model

In [ ]:
lstm_model = models.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    layers.LSTM(32),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)   # Regression output
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm_model.summary()

### Step 5: Train the LSTM Model

In [ ]:
lstm_history = lstm_model.fit(
    X_tr, y_tr,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

### Step 6: Predict and Evaluate

In [ ]:
y_pred_scaled = lstm_model.predict(X_te)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_true = scaler.inverse_transform(y_te.reshape(-1,1))

from sklearn.metrics import mean_absolute_error, mean_squared_error
print(f'MAE:  {mean_absolute_error(y_true, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}')

### Step 7: Visualize LSTM Performance

In [ ]:
# Prediction vs Actual
plt.figure(figsize=(12, 4))
plt.plot(y_true, label='Actual')
plt.plot(y_pred, label='Predicted', linestyle='--')
plt.title('LSTM Time Series Prediction')
plt.xlabel('Time Steps')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

# Loss curve
plt.figure(figsize=(7, 4))
plt.plot(lstm_history.history['loss'],     label='Train Loss')
plt.plot(lstm_history.history['val_loss'], label='Val Loss')
plt.title('LSTM Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()